# ⛪🎬 VC ChurchEdition - Criador de Vídeos Curtos para Igrejas

Este projeto é uma ferramenta incrível para **extrair os melhores momentos de pregações e cultos** e transformá-los automaticamente em vídeos curtos (Reels, TikTok, Shorts) com legendas dinâmicas.

Ele roda diretamente na nuvem (Google Colab) salvando os vídeos no seu Google Drive!


## 📖 Como Usar (Muito Simples!)

Apenas clique no botão de **'Play' (▶️)** nos dois passos abaixo. Espere o primeiro terminar para clicar no segundo.

**Passo 1: Preparar o Ambiente**
Vai pedir permissão para acessar seu Google Drive (é lá que os vídeos serão salvos) e instalará os arquivos necessários.

**Passo 2: Iniciar o Criador de Vídeos**
Quando terminar de carregar, ele vai gerar um link azul (`public URL`). Clique nele para abrir a ferramenta na mesma hora!

**Onde ficam meus vídeos prontos?**
Eles aparecerão magicamente no seu Google Drive, em uma pasta chamada `Cortes_Igreja`.


In [ ]:
#@title 🛠️ Passo 1: Preparar o Ambiente (Conectar Drive e Instalar)
#@markdown ### Repositorio GitHub
repo_url = "https://github.com/Kadenai/VC-ChurchEdition.git" #@param {type:"string"}
branch = "main" #@param {type:"string"}
#@markdown ### Pasta de saida no Drive (videos gerados)
#@markdown Os videos processados serao salvos nesta pasta do seu Drive.
output_folder = "Cortes_Igreja" #@param {type:"string"}

import os, shutil, subprocess
from google.colab import drive

print("Montando Google Drive...")
drive.mount('/content/drive')

project_dir = "/content/ViralCutter"
output_drive = f"/content/drive/MyDrive/{output_folder}"

# Clonar ou atualizar projeto
if os.path.exists(os.path.join(project_dir, ".git")):
    print("Atualizando projeto pelo GitHub...")
    subprocess.run(["git", "-C", project_dir, "fetch", "origin", branch], check=True)
    subprocess.run(["git", "-C", project_dir, "reset", "--hard", f"origin/{branch}"], check=True)
elif os.path.exists(project_dir):
    print("Removendo pasta antiga sem Git...")
    shutil.rmtree(project_dir)
    print("Clonando projeto pelo GitHub...")
    subprocess.run(["git", "clone", "--branch", branch, "--depth", "1", repo_url, project_dir], check=True)
else:
    print("Clonando projeto pelo GitHub...")
    subprocess.run(["git", "clone", "--branch", branch, "--depth", "1", repo_url, project_dir], check=True)

# Limpar artefatos locais que nao fazem sentido no Colab
cleaned = 0
for root, dirs, files in os.walk(project_dir):
    for d in list(dirs):
        if d in ('__pycache__', '.venv', '.claude'):
            shutil.rmtree(os.path.join(root, d), ignore_errors=True)
            cleaned += 1
if cleaned:
    print(f"{cleaned} pasta(s) temporaria(s) removida(s).")

# Vincular pasta VIRALS ao Google Drive (saida persistente)
os.makedirs(output_drive, exist_ok=True)
virals_local = os.path.join(project_dir, "VIRALS")
if os.path.islink(virals_local):
    os.unlink(virals_local)
elif os.path.exists(virals_local):
    shutil.rmtree(virals_local)
os.symlink(output_drive, virals_local)

print(f"\nProjeto pronto em: {project_dir}")
print(f"Videos serao salvos em: Google Drive > {output_folder}")
print("\nExecute a proxima celula para instalar as dependencias!")


import os
import subprocess
from IPython.display import clear_output

%cd /content/ViralCutter

print("⏳ Iniciando instalação completa...")
print("(Isso pode levar 5-10 minutos na primeira vez)\n")

# 1. UV + Drivers do sistema
print("📦 [1/10] Instalando UV e drivers do sistema...")
subprocess.run(['pip', 'install', 'uv'], check=True)
print('⚙️ Atualizando repositórios apt (ignora falhas)...')
subprocess.run('sudo apt update -y || true', shell=True)
print('⚙️ Instalando ffmpeg, xvfb e fontes...')
subprocess.run('sudo apt install -y ffmpeg xvfb fonts-montserrat || sudo apt install -y ffmpeg xvfb', shell=True, check=True)
print('⚙️ Instalando drivers de aceleração adicionais...')
subprocess.run('sudo apt install -y libcudnn8 || true', shell=True)
print('⚙️ Garantindo que fontes Montserrat estejam instaladas...')
try:
    subprocess.run('sudo mkdir -p /usr/share/fonts/truetype/montserrat', shell=True)
    subprocess.run('sudo wget -q -O /usr/share/fonts/truetype/montserrat/Montserrat-Regular.ttf https://github.com/google/fonts/raw/main/ofl/montserrat/Montserrat-Regular.ttf', shell=True)
    subprocess.run('sudo wget -q -O /usr/share/fonts/truetype/montserrat/Montserrat-Bold.ttf https://github.com/google/fonts/raw/main/ofl/montserrat/Montserrat-Bold.ttf', shell=True)
    subprocess.run('sudo fc-cache -f || true', shell=True)
    print('✨ Fontes Montserrat configuradas!')
except Exception as e:
    print(f'⚠️ Aviso ao configurar fontes: {e}')

# 2. Ambiente Virtual
print("🐍 [2/10] Criando ambiente virtual...")
subprocess.run(['uv', 'venv', '.venv'], check=True)

# 3. PyTorch + CUDA (cu121 — compatível com Colab T4)
print("🔥 [3/10] Instalando PyTorch 2.3.1 + CUDA 12.1...")
cmd_torch = (
    "uv pip install --python .venv "
    "torch==2.3.1+cu121 torchvision==0.18.1+cu121 torchaudio==2.3.1+cu121 "
    "--index-url https://download.pytorch.org/whl/cu121"
)
subprocess.run(cmd_torch, shell=True, check=True)

# 4. WhisperX (do GitHub — versão mais recente com correções de alinhamento)
print("🎤 [4/10] Instalando WhisperX (GitHub)...")
subprocess.run("uv pip install --python .venv git+https://github.com/m-bain/whisperx.git", shell=True, check=True)

# 5. Dependências do requirements.txt
print("📋 [5/10] Instalando requirements.txt...")
subprocess.run("uv pip install --python .venv -r requirements.txt", shell=True, check=True)

# 6. Pacotes extras e correções de compatibilidade
print("🔧 [6/10] Instalando pacotes extras...")
extras = [
    "uv pip install --python .venv yt-dlp pytubefix",
    "uv pip install --python .venv google-generativeai",
    "uv pip install --python .venv pandas",
    "uv pip install --python .venv onnxruntime-gpu",
    "uv pip install --python .venv transformers==4.46.3 'accelerate>=0.26.0'",
]
for cmd in extras:
    subprocess.run(cmd, shell=True, check=True)

# 7. Re-pin PyTorch (garante que nenhuma dependência atualizou sem querer)
print("🔨 [7/10] Re-fixando versão do PyTorch...")
subprocess.run(cmd_torch, shell=True, check=True)

# 8. Travar Numpy e setuptools
print("🔨 [8/10] Travando Numpy...")
subprocess.run("uv pip install --python .venv 'numpy<2.0' setuptools==69.5.1", shell=True, check=True)

# 9. Correção de Visão Computacional (MediaPipe + InsightFace)
print("🔨 [9/10] Corrigindo MediaPipe + InsightFace...")
subprocess.run("uv pip uninstall --python .venv mediapipe protobuf flatbuffers", shell=True)
subprocess.run("uv pip install --python .venv 'mediapipe>=0.10.0' 'protobuf>=3.20,<5.0' 'flatbuffers>=2.0'", shell=True, check=True)
subprocess.run("uv pip install --python .venv insightface onnxruntime-gpu", shell=True, check=True)

# 10. Display virtual (para OpenCV/MediaPipe em modo headless)
print("🖥️ [10/10] Configurando display virtual...")
os.system('Xvfb :1 -screen 0 2560x1440x8 &')
os.environ['DISPLAY'] = ':1.0'

clear_output()
print("✅ Instalação Completa!")
print("━" * 42)
print("  • PyTorch 2.3.1 + CUDA 12.1      ✅")
print("  • WhisperX (GitHub latest)        ✅")
print("  • Transformers 4.46.3             ✅")
print("  • Numpy < 2.0                     ✅")
print("  • MediaPipe + InsightFace          ✅")
print("  • FastAPI + Gradio + Extras        ✅")
print("  • deep-translator + tqdm           ✅")
print("━" * 42)
print("\n▶️ Execute a próxima célula para iniciar a WebUI!")

In [ ]:
#@title 🚀 Passo 2: Executar ViralCutter WebUI
import os

%cd /content/ViralCutter

# Display virtual (caso o kernel tenha reiniciado)
os.system('Xvfb :1 -screen 0 2560x1440x8 &')
os.environ['DISPLAY'] = ':1.0'
os.environ['MPLBACKEND'] = 'Agg'
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
os.environ.setdefault('VC_LIBRARY_MAX_CARDS', '24')

print("🚀 Iniciando ViralCutter WebUI...")
print("⏳ Aguarde o link público (public URL) aparecer abaixo...")
print("⚠️ Ignore avisos de 'UserWarning'\n")

!/content/ViralCutter/.venv/bin/python webui/app.py --colab

---

## 🙌 Créditos e Agradecimentos

Este projeto é um *fork* direcionado a Igrejas.
A ferramenta base original ([ViralCutter](https://github.com/RafaelGodoyEbert/ViralCutter)) foi desenvolvida por **Rafael Godoy**.
Todo o mérito da engenharia por trás do processamento dos vídeos é devido ao criador original!

Apoie o projeto original:
[![GitHub](https://img.shields.io/badge/github-%23121011.svg?style=for-the-badge&logo=github&logoColor=white)](https://github.com/RafaelGodoyEbert/ViralCutter)
